# Heart Disease Model Training & Comparison
This notebook loads the dataset, applies data cleaning and preprocessing, trains multiple models, compares them on all criteria (Accuracy, Precision, Recall, F1-Score, ROC-AUC), and saves the best pipeline.

In [6]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix
)

# Models to compare
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

import warnings
warnings.filterwarnings('ignore')

In [7]:
BASE_DIR = os.path.abspath('..')
DATA_DIR = os.path.join(BASE_DIR, 'Data')
MODEL_DIR = os.path.join(BASE_DIR, 'Model')
os.makedirs(MODEL_DIR, exist_ok=True)

file_path = os.path.join(DATA_DIR, 'heart_disease.csv')
df = pd.read_csv(file_path)
print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['target'].value_counts()}\n")
df.head()

Dataset shape: (1025, 14)
Class distribution:
target
1    526
0    499
Name: count, dtype: int64



,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [8]:
# Basic cleaning
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,1,1,140,221,0,1,164,1,0.0,2,0,2,1
1021,60,1,0,125,258,0,0,141,1,2.8,1,1,3,0
1022,47,1,0,110,275,0,0,118,1,1.0,1,1,2,0
1023,50,0,0,110,254,0,0,159,0,0.0,2,0,2,1


In [9]:
X = df.drop(columns=['target'])
y = df['target'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])


In [10]:
pipelines = {
    'Logistic Regression': Pipeline([('preprocessor', preprocessor), ('model', LogisticRegression(max_iter=1000, random_state=42))]),
    'Gaussian Naive Bayes': Pipeline([('preprocessor', preprocessor), ('model', GaussianNB())]),
    'Random Forest': Pipeline([('preprocessor', preprocessor), ('model', RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42))]),
    'Gradient Boosting': Pipeline([('preprocessor', preprocessor), ('model', GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=4, min_samples_split=5, min_samples_leaf=2, random_state=42))]),
    # 'SVM (RBF)': Pipeline([('preprocessor', preprocessor), ('model', SVC(kernel='rbf', probability=True, random_state=42))]),
    # 'KNN': Pipeline([('preprocessor', preprocessor), ('model', KNeighborsClassifier(n_neighbors=7))]),
    'Decision Tree': Pipeline([('preprocessor', preprocessor), ('model', DecisionTreeClassifier(max_depth=5, min_samples_split=5, min_samples_leaf=2, random_state=42))]),
    'AdaBoost': Pipeline([('preprocessor', preprocessor), ('model', AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42))]),
}


In [11]:
print('=' * 70)
print('  MODEL COMPARISON RESULTS')
print('=' * 70)

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, pipeline in pipelines.items():
    try:
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        if hasattr(pipeline['model'], 'predict_proba'):
            y_proba = pipeline.predict_proba(X_test)[:, 1]
        else:
            y_proba = y_pred
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        auc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) == 2 else 0
        
        cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()
        
        results.append({
            'Model': name,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-Score': f1,
            'ROC-AUC': auc,
            'CV Mean': cv_mean,
            'CV Std': cv_std,
        })
    except Exception as e:
        print(f"Error training {name}: {e}")


  MODEL COMPARISON RESULTS


In [12]:
results_df = pd.DataFrame(results)
results_df['Composite'] = (
    results_df['F1-Score'] * 0.30 +
    results_df['ROC-AUC'] * 0.30 +
    results_df['Accuracy'] * 0.15 +
    results_df['Precision'] * 0.10 +
    results_df['Recall'] * 0.10 +
    results_df['CV Mean'] * 0.05
)

results_df = results_df.sort_values('Composite', ascending=False).reset_index(drop=True)

print('\n\n' + '=' * 70)
print('  FINAL COMPARISON TABLE (sorted by composite score)')
print('=' * 70)
print(results_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

best_model_name = results_df.iloc[0]['Model']
print(f"\n{'★' * 50}")
print(f"  BEST MODEL: {best_model_name}")
print(f"  Composite Score: {results_df.iloc[0]['Composite']:.4f}")
print(f"{'★' * 50}")




  FINAL COMPARISON TABLE (sorted by composite score)
               Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC  CV Mean  CV Std  Composite
   Gradient Boosting    1.0000     1.0000  1.0000    1.0000   1.0000   0.9841  0.0157     0.9992
       Random Forest    0.9902     0.9813  1.0000    0.9906   0.9994   0.9744  0.0098     0.9924
            AdaBoost    0.8780     0.8636  0.9048    0.8837   0.9373   0.8488  0.0226     0.8973
       Decision Tree    0.8732     0.8624  0.8952    0.8785   0.9326   0.8927  0.0179     0.8947
 Logistic Regression    0.8098     0.7619  0.9143    0.8312   0.9298   0.8427  0.0170     0.8595
Gaussian Naive Bayes    0.8293     0.8070  0.8762    0.8402   0.9043   0.8354  0.0329     0.8578

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  BEST MODEL: Gradient Boosting
  Composite Score: 0.9992
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


In [13]:
best_pipeline = pipelines[best_model_name]
y_pred_best = best_pipeline.predict(X_test)

print(f"\n{'=' * 70}")
print(f"  DETAILED REPORT: {best_model_name}")
print(f"{'=' * 70}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

model_path = os.path.join(MODEL_DIR, 'heart_disease_pipeline.pkl')
joblib.dump(best_pipeline, model_path)
print(f"\n✅ Saved best model ({best_model_name}) pipeline to {model_path}")


  DETAILED REPORT: Gradient Boosting

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00       105

    accuracy                           1.00       205
   macro avg       1.00      1.00      1.00       205
weighted avg       1.00      1.00      1.00       205


✅ Saved best model (Gradient Boosting) pipeline to c:\Users\Prince\OneDrive\Desktop\Health Risk Prediction\Model\heart_disease_pipeline.pkl
